# B1 GRPO training on Qwen3-1.7B (Workstream B Phase 3)

Trains the **B1 flat-agent baseline** with [Unsloth](https://docs.unsloth.ai/get-started/reinforcement-learning-rl-guide) + [TRL GRPO](https://huggingface.co/docs/trl/main/en/grpo_trainer) against the deployed CrisisWorldCortex HF Space env.

**One-shot run:** `Runtime → Run all` on a fresh Colab T4. 300 training steps, ~30 minutes wall-clock. Saves the trained LoRA adapter to your HF Hub at the end.

**Reward source:** the Phase-1-fixed `outer_reward` (range `[-1.0, 1.0]`, signal-quality gates passed). Single-step GRPO — each rollout = one env reset + one env step.

**Prereqs:**
1. Colab Secrets has `HF_TOKEN` set (Tools → Secrets, name = `HF_TOKEN`, value = your `hf_xxx` token with write access).
2. The HF Space `Angshuman28/CrisisWorldCortex` is running with the post-Phase-1 reward.


## 1. Install dependencies

Unsloth pulls a custom torch + vllm + xformers stack tuned for free Colab T4. Pin trl to a version compatible with `GRPOTrainer` (>=0.14).

In [ ]:
%%capture
!pip install --upgrade pip
!pip install unsloth vllm
!pip install --upgrade --no-deps "trl>=0.14" peft accelerate bitsandbytes
!pip install pydantic openenv huggingface_hub matplotlib

## 2. Authenticate with Hugging Face

Reads `HF_TOKEN` from Colab Secrets. Falls back to interactive login if not set.

In [ ]:
import os

try:
    from google.colab import userdata

    HF_TOKEN = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
except Exception:
    from huggingface_hub import login

    login()
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

assert HF_TOKEN, "HF_TOKEN is required (set in Colab Secrets or via login())"
print("HF auth OK")

## 3. Clone CrisisWorldCortex and install

Pulls the deployed HF Space's repo and installs locally. Provides the `CrisisworldcortexEnv` HTTP client + the `baselines.flat_agent` system prompt + parser used by B1.

In [ ]:
%%capture
!rm -rf /content/CrisisWorldCortex
!git clone https://huggingface.co/spaces/Angshuman28/CrisisWorldCortex /content/CrisisWorldCortex
%cd /content/CrisisWorldCortex
!pip install -e .

In [ ]:
# Sanity: imports resolve, env client constructs.
import sys

sys.path.insert(0, "/content/CrisisWorldCortex")

from baselines.flat_agent import (
    build_system_prompt,
    parse_action,
    parse_failure_marker,
    serialize_observation,
)
from CrisisWorldCortex import CrisisworldcortexAction, CrisisworldcortexObservation
from CrisisWorldCortex.client import CrisisworldcortexEnv

print("CrisisWorld imports OK")

## 4. Load Qwen3-1.7B with LoRA via Unsloth

Qwen3-1.7B fits comfortably on a T4 with 4-bit quantization. LoRA rank 32 — enough to learn the JSON-action format and modest policy improvements; cheap to merge.

In [ ]:
import torch
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 4096
MODEL_NAME = "unsloth/Qwen3-1.7B"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    fast_inference=True,  # vLLM-backed generate, required by GRPOTrainer
    max_lora_rank=32,
    gpu_memory_utilization=0.6,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=64,
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("Model + LoRA loaded")

## 5. Connect to the deployed CrisisWorld env

Uses the public HF Space URL. Each rollout calls `env.reset()` then `env.step(action)` once.

In [ ]:
ENV_URL = "https://angshuman28-crisisworldcortex.hf.space"
TASKS = ("outbreak_easy", "outbreak_medium", "outbreak_hard")
EPISODE_TICKS = 12


def make_env() -> CrisisworldcortexEnv:
    return CrisisworldcortexEnv(base_url=ENV_URL)


_test_env = make_env()
_obs = _test_env.reset(task_name="outbreak_easy", seed=0, max_ticks=EPISODE_TICKS)
print(f"Env OK. Initial tick={_obs.tick}, regions={[r.region for r in _obs.regions]}")

## 6. Build the prompt dataset and reward function

Each example in the dataset is a `(task, seed)` pair. The reward function:
1. Resets the env to that `(task, seed)`.
2. Parses the model's completion as a `OuterActionPayload`.
3. Submits to the env, returns `obs.reward` (post-Phase-1 range `[-1, 1]`).
4. On parse failure, submits `parse_failure_marker()` so the §19 -1.0 + terminate contract fires.

In [ ]:
import random

from datasets import Dataset

SYSTEM_PROMPT = build_system_prompt()


def build_user_prompt(obs: CrisisworldcortexObservation) -> str:
    return serialize_observation(obs)


def make_chat_prompt(obs: CrisisworldcortexObservation) -> str:
    return tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(obs)},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )


rng = random.Random(0)
_seed_pool = []
for task in TASKS:
    for seed in range(50):
        _seed_pool.append({"task": task, "seed": seed})
rng.shuffle(_seed_pool)

_prompts = []
_meta = []
for entry in _seed_pool:
    env = make_env()
    obs = env.reset(task_name=entry["task"], seed=entry["seed"], max_ticks=EPISODE_TICKS)
    _prompts.append(make_chat_prompt(obs))
    _meta.append(entry)

train_dataset = Dataset.from_dict(
    {
        "prompt": _prompts,
        "task": [m["task"] for m in _meta],
        "seed": [m["seed"] for m in _meta],
    }
)
print(f"Dataset built: {len(train_dataset)} examples")

In [ ]:
def crisisworld_reward(
    prompts: list[str],
    completions: list[str],
    task: list[str],
    seed: list[int],
    **_kwargs,
) -> list[float]:
    """GRPO reward function: one env step per (prompt, completion) pair.

    Reward source: Phase-1-fixed env outer_reward in [-1, 1].
    Parse failure → submits parse_failure_marker → r_policy = -1.0 + terminate.
    """
    rewards: list[float] = []
    for completion, t, s in zip(completions, task, seed):
        env = make_env()
        env.reset(task_name=t, seed=int(s), max_ticks=EPISODE_TICKS)
        action_payload = parse_action(completion)
        if action_payload is None:
            action_payload = parse_failure_marker()
        try:
            result = env.step(CrisisworldcortexAction(action=action_payload))
            reward = result.observation.reward if hasattr(result, "observation") else result.reward
            rewards.append(float(reward) if reward is not None else 0.0)
        except Exception as exc:
            print(f"[WARN] env.step failed task={t} seed={s}: {exc}")
            rewards.append(-1.0)
    return rewards

## 7. GRPO training

300 steps × group size 4 = 1200 rollouts. Each rollout is one HF Space round-trip (~1s) — total ~20–30 min on T4.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

MAX_TRAIN_STEPS = 300
GROUP_SIZE = 4
MAX_PROMPT_LEN = 2048
MAX_COMPLETION_LEN = 512

training_args = GRPOConfig(
    output_dir="/content/b1_grpo_output",
    learning_rate=5e-6,
    per_device_train_batch_size=GROUP_SIZE,
    gradient_accumulation_steps=1,
    num_generations=GROUP_SIZE,
    max_prompt_length=MAX_PROMPT_LEN,
    max_completion_length=MAX_COMPLETION_LEN,
    max_steps=MAX_TRAIN_STEPS,
    save_steps=100,
    logging_steps=5,
    report_to="none",
    bf16=True,
    optim="adamw_8bit",
    temperature=0.8,
    use_vllm=True,
    vllm_mode="colocate",
    seed=42,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[crisisworld_reward],
    args=training_args,
    train_dataset=train_dataset,
)
print("GRPOTrainer constructed; starting train()...")

In [ ]:
trainer.train()

## 8. Save the trained LoRA adapter to HF Hub

Pushes to `<your-username>/crisisworld-b1-grpo-qwen3-1p7b`. Change the repo name below if you want a different namespace.

In [ ]:
from huggingface_hub import HfApi

HUB_REPO = "Angshuman28/crisisworld-b1-grpo-qwen3-1p7b"

model.save_pretrained("/content/b1_grpo_lora")
tokenizer.save_pretrained("/content/b1_grpo_lora")

api = HfApi()
api.create_repo(HUB_REPO, exist_ok=True, repo_type="model", private=False, token=HF_TOKEN)
api.upload_folder(
    folder_path="/content/b1_grpo_lora",
    repo_id=HUB_REPO,
    repo_type="model",
    token=HF_TOKEN,
)
print(f"Saved to https://huggingface.co/{HUB_REPO}")

## 9. Eval: trained adapter vs base model on 3 tasks

Runs a single full episode (12 ticks) per task, per model. Reports cumulative reward.

In [ ]:
def _hf_chat(model_inst, tokenizer_inst, system: str, user: str, max_new_tokens: int = 256) -> str:
    prompt = tokenizer_inst.apply_chat_template(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer_inst(prompt, return_tensors="pt").to(model_inst.device)
    with torch.no_grad():
        out = model_inst.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
        )
    return tokenizer_inst.decode(out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True)


def run_one_episode(model_inst, tokenizer_inst, task: str, seed: int) -> float:
    env = make_env()
    obs = env.reset(task_name=task, seed=seed, max_ticks=EPISODE_TICKS)
    cumulative = 0.0
    for tick in range(EPISODE_TICKS):
        completion = _hf_chat(model_inst, tokenizer_inst, SYSTEM_PROMPT, serialize_observation(obs))
        action = parse_action(completion) or parse_failure_marker()
        result = env.step(CrisisworldcortexAction(action=action))
        obs = result.observation if hasattr(result, "observation") else result
        reward = obs.reward if obs.reward is not None else 0.0
        cumulative += reward
        if obs.done:
            break
    return cumulative


FastLanguageModel.for_inference(model)
trained_results = {t: run_one_episode(model, tokenizer, t, seed=0) for t in TASKS}
print(f"Trained model cumulative reward per task: {trained_results}")

In [ ]:
# Reload base Qwen3-1.7B (no LoRA) for the comparison.
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    fast_inference=False,
)
FastLanguageModel.for_inference(base_model)
base_results = {t: run_one_episode(base_model, base_tokenizer, t, seed=0) for t in TASKS}
print(f"Base model cumulative reward per task: {base_results}")

## 10. Plot eval comparison

Bar chart: trained vs base, cumulative episode reward by task.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

task_names = list(TASKS)
trained_vals = [trained_results[t] for t in task_names]
base_vals = [base_results[t] for t in task_names]

x = np.arange(len(task_names))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width / 2, base_vals, width, label="Base Qwen3-1.7B")
ax.bar(x + width / 2, trained_vals, width, label="GRPO-trained Qwen3-1.7B")
ax.set_xticks(x)
ax.set_xticklabels(task_names)
ax.set_ylabel("Cumulative episode reward")
ax.set_title("B1 GRPO: trained vs base")
ax.legend()
ax.axhline(0.0, linestyle=":", color="grey")
plt.tight_layout()
plt.show()